# 02 · Exploratory Data Analysis
Behavioral patterns, temporal trends, and NLP insights.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, plotly.express as px
from collections import Counter
import re, warnings; warnings.filterwarnings("ignore")

df = pd.read_csv("../data/whatsapp_cleaned_data.csv")
df["datetime"] = pd.to_datetime(df["date"]+" "+df["time"], format="%m/%d/%y %I:%M %p", errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)
df["hour"] = df["datetime"].dt.hour
df["day"]  = df["datetime"].dt.day_name()
df["month"]= df["datetime"].dt.month_name()
df["msg_length"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()
print(df.shape); df.head()

In [ ]:
# Overview metrics
print(f"Messages: {len(df):,}")
print(f"Users: {df['user'].nunique()}")
print(f"Date range: {df['datetime'].min().date()} to {df['datetime'].max().date()}")
df["user"].value_counts().head(10).plot(kind="bar", title="Top 10 Active Users", figsize=(10,4))
plt.tight_layout(); plt.show()

In [ ]:
# Hourly distribution
df["hour"].value_counts().sort_index().plot(kind="bar", figsize=(12,4), title="Hourly Activity")
plt.xlabel("Hour"); plt.ylabel("Messages"); plt.tight_layout(); plt.show()

In [ ]:
# Activity heatmap
pivot = df.groupby(["day","hour"]).size().unstack(fill_value=0)
day_order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
pivot = pivot.reindex([d for d in day_order if d in pivot.index])
plt.figure(figsize=(14,5))
sns.heatmap(pivot, cmap="Blues", linewidths=.3, linecolor="#0f1117")
plt.title("Message Activity Heatmap"); plt.tight_layout(); plt.show()

In [ ]:
# Response time analysis
df["prev_user"] = df["user"].shift(1)
df["prev_time"] = df["datetime"].shift(1)
df["response_time"] = np.where(
    df["user"] != df["prev_user"],
    (df["datetime"]-df["prev_time"]).dt.total_seconds()/60, np.nan)
resp = df[df["response_time"].notna() & (df["response_time"]>0) & (df["response_time"]<240)]
avg_resp = resp.groupby("user")["response_time"].mean().sort_values().head(10)
avg_resp.plot(kind="barh", figsize=(10,5), title="Top 10 Fastest Responders (min)")
plt.tight_layout(); plt.show()

In [ ]:
# Sentiment analysis with VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
df["sentiment"] = df["message"].apply(lambda x: sia.polarity_scores(str(x))["compound"])
df["sentiment_label"] = df["sentiment"].apply(lambda s: "Positive" if s>.05 else ("Negative" if s<-.05 else "Neutral"))
print(df["sentiment_label"].value_counts())
df["sentiment_label"].value_counts().plot(kind="pie", autopct="%1.1f%%", figsize=(6,6), title="Sentiment Distribution")
plt.tight_layout(); plt.show()

In [ ]:
# Top keywords
STOP = {"the","a","and","or","is","it","i","you","we","to","of","in","for","on","at",
        "ka","ki","ke","hai","hain","bhi","kya","nhi","nahi","aur","yeh","toh"}
text = " ".join(df["message"].astype(str)).lower()
words = [w for w in re.findall(r"\b[a-zA-Z]{3,}\b",text) if w not in STOP]
top30 = Counter(words).most_common(30)
pd.DataFrame(top30, columns=["word","count"]).set_index("word").plot(kind="barh",figsize=(10,10),title="Top 30 Keywords")
plt.tight_layout(); plt.show()